# Week 2: Neural Networks, Backpropagation, and SGD

This week compresses the neural-network material from Fall 2025 Weeks 3 and 4 into one summer lecture. We will move from the conceptual pieces of a neural network to a minimal NumPy implementation of backpropagation, then speed up learning with stochastic gradient descent and cross-entropy loss.

By the end of the notebook, you should be able to:

1. describe the forward pass, loss computation, backpropagation, and gradient descent loop;
2. explain why backpropagation is much more efficient than finite-difference gradients for neural networks;
3. implement and train a small fully-connected neural network;
4. use mini-batches and cross-entropy to train faster on MNIST.


## Neural Networks and Backpropagation

The approach to making neural networks learn is very similar to the gradient-based learning approach we used previously, but the scale of large neural networks requires us to adjust certain parts of it. There are four main bits of code we need to make neural nets work:

1. **Forward pass**: feeds a datapoint into a neural network and processes it by computing the outputs of each layer of artificial neurons until we reach the output layer and compute the loss

2. **Loss**: after feeding in some labeled data, we can compute the loss function measuring the network's training error.

3. **Backpropagation (Backprop)**: uses the chain rule from calculus in a systematic way to find the gradient of the loss function.

4. **Gradient descent**: at each iteration, nudge the weights in the opposite direction of the gradient.

#### Note: *Stochastic* Gradient Descent (SGD)

We will discuss and add SGD later in this lecture. It speeds up gradient descent by making weight updates more frequently--based on samples of the training data rather than the entire dataset. SGD is critical for large neural networks to operate properly, but we will avoid scaling up too much this lecture.

### The Forward Pass

The forward pass is simple, we just follow the simple rules of the individual neurons, but we have to do it **many** times, which we can automate with matrix multiplication.

### Loss

The loss function can be computed quite simply in the same way as we did last week. However, we may want to add weight decay or other features to the loss function or we may want to use a different loss function entirely.

### Backpropagation

In the past, we approximated the gradient by computing the loss functions many times, perturbed in each dimension to compute

$$\nabla L(w)\approx\left(\frac{L(w+he_0)-L(w)}{h}, ..., \frac{L(w+he_d)-L(w)}{h}\right)$$

for some small value $h$, but this meant we had to compute the loss function $d+1$ times for **each** iteration in gradient descent, which required us to compute the outputs of the model for the whole dataset many, many times. This was fine for the linear regression models we have considered, but it would be too computationally expensive to feed the whole dataset through a neural network so many times.

As a result, we instead need another method to compute gradients. It turns out, we can systematically apply the chain rule to get formulas for the partial derivatives of the loss function with respect to each weight in the neural network. Backpropagation does just this.

### Gradient Descent

As we have seen before, gradient descent is an iterative learning algorithm that aims to minimize the loss function. It iteratively nudges the weights in the opposite direction of the gradients of the loss function.

### Backpropagation Notes

See the introductory notes on backpropagation in the PDF lecture notes in Canvas. The implementation below turns those chain-rule formulas into code.

## From Backpropagation to Faster Training

### Implementing Backpropagation

Below, we implement a fully-connected feedforward neural network with random weight initialization, customizable architecture, and gradient descent with use of backpropagation for computing the gradients. We will then test it on some examples.

The best way to understand how this all works is to actually implement it with minimal external libraries. One exception is `NumPy`, which at least automates some of the basic mathematical operations we need to do.

In [ ]:
import numpy as np

Next, we will implement the feedforward neural network. The code is partly based on an implementation from *Deep Learning for Computer Vision with Python* by Adrian Rosebrock.

In [ ]:
class FeedforwardNeuralNetwork:
    
    # input a vector [a, b, c, ...] with the number of nodes in each layer
    def __init__(self, layers, alpha = 0.1):
        
        # list of weight matrices between layers
        self.W = []
        
        # network architecture will be a vector of numbers of nodes for each layer
        self.layers = layers
        
        # learning rate
        self.alpha = alpha
        
        # initialize the weights (randomly) -- this is our initial guess for gradient descent
        
        # initialize the weights between layers (up to the next-to-last one) as normal random variables
        for i in np.arange(0, len(layers) - 2):
            self.W.append(np.random.randn(layers[i] + 1, layers[i + 1] + 1))
            
        # initialize weights between the last two layers (we don't want bias for the last one)
        self.W.append(np.random.randn(layers[-2] + 1, layers[-1]))
        
    # define the sigmoid activation
    def sigmoid(self, x):
        return 1.0 / (1 + np.exp(-x))
    
    # define the sigmoid derivative (where z is the output of a sigmoid)
    def sigmoidDerivative(self, z):
        return z * (1 - z)
    
    # fit the model
    def fit(self, X, y, epochs = 10000, update = 1000):
        # add a column of ones to the end of X
        X = np.hstack((X, np.ones([X.shape[0],1])))

        for epoch in np.arange(0,epochs):

            # feed forward, backprop, and weight update
            for (x, target) in zip(X, y):
                
                # make a list of output activations from the first layer
                # (just the original x values)
                A = [np.atleast_2d(x)]
                
                # feed forward
                for layer in np.arange(0, len(self.W)):
                    
                    # feed through one layer and apply sigmoid activation
                    net = A[layer].dot(self.W[layer])
                    out = self.sigmoid(net)
                    
                    # add our network output to the list of activations
                    A.append(out)
                    
                # backpropagation
                error = A[-1] - target
                
                # term proportional to the gradient
                D = [error * self.sigmoidDerivative(A[-1])]
                
                # loop backwards over the layers to build up deltas
                for layer in np.arange(len(A) - 2, 0, -1):
                    delta = D[-1].dot(self.W[layer].T)
                    delta = delta * self.sigmoidDerivative(A[layer])
                    D.append(delta)
                    
                # reverse the deltas since we looped in reverse
                D = D[::-1]
                
                # weight update
                for layer in np.arange(0, len(self.W)):
                    self.W[layer] -= self.alpha * A[layer].T.dot(D[layer])
                    
            # print a status update
            if (epoch + 1) % update == 0:
                loss = self.computeLoss(X,y)
                print('Epoch =', epoch + 1, 'loss = ', loss)
                
    def predict(self, X, addOnes = True):
        # initialize data, be sure it's the right dimension
        p = np.atleast_2d(X)
        
        # add a column of 1s for bias
        if addOnes:
            p = np.hstack((p, np.ones([X.shape[0],1])))
        
        # feed forward!
        for layer in np.arange(0, len(self.W)):
            p = self.sigmoid(np.dot(p, self.W[layer]))
         
        # return the predictions
        return p
    
    def computeLoss(self, X, y):
        # initialize data, be sure it's the right dimension
        y = np.atleast_2d(y)
        
        # feed the datapoints through the network to get predicted outputs
        predictions = self.predict(X, addOnes = False)
        
        # compute the sum of squared errors loss function
        loss = np.sum((predictions - y)**2) / 2.0
        
        return loss

Let's check the variables of the model.

In [ ]:
model = FeedforwardNeuralNetwork([2, 2, 1])
vars(model)

### Example: XOR function

Let's run the code to try to learn the exclusive 'or' function, i.e. XOR.

In [ ]:
import random

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

model.fit(X,y,50000,1000)

In [ ]:
model.predict(X)

Note these are very close to 0, 1, 1, 0 -- the correct classifications for the XOR function. This neural net can classify this nonlinear problem of XOR.

### Example: Tiny MNIST

In [ ]:
# load some more libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn import datasets
from tensorflow.keras.utils import to_categorical

Let's try to classify a tiny version of the MNIST data shrunk down to 8-by-8 pixel images.

In [ ]:
### CLASSIFY MNIST PICTURES

# 'digits' is a tiny version of MNIST in sklearn.datasets has only 8 x 8 pixels
# with grayscale values from 0 to 16
digits = datasets.load_digits()
data = digits.data.astype("float")
print("Samples: {}, Dimension: {}".format(data.shape[0], data.shape[1]))

X = data

Y = digits.target

# randomly choose 75% of the data to be the training set and 25% for the testing set
trainX, testX, trainY, testY = train_test_split(X, Y, test_size = 0.25)

# we need to turn the labels into 1-hot representations
trainY = to_categorical(trainY)
testY = to_categorical(testY)

# fit the model to the training data
model = FeedforwardNeuralNetwork([64, 16, 10])
model.fit(trainX, trainY, 1000, 100)

# print the classification performance
print("Training set accuracy")

predictedY = model.predict(trainX)
predictedY = predictedY.argmax(axis=1)

trainY = trainY.argmax(axis=1)

print(classification_report(trainY, predictedY))

print("Test set accuracy")

predictedY = model.predict(testX)
predictedY = predictedY.argmax(axis=1)

testY = testY.argmax(axis=1)

print(classification_report(testY, predictedY))

Let's try the same with two hidden layers.

In [ ]:
### CLASSIFY MNIST PICTURES

# 'digits' is a tiny version of MNIST in sklearn.datasets has only 8 x 8 pixels
# with grayscale values from 0 to 16
digits = datasets.load_digits()
data = digits.data.astype("float")
print("Samples: {}, Dimension: {}".format(data.shape[0], data.shape[1]))

X = data/16.0

Y = digits.target

# randomly choose 75% of the data to be the training set and 25% for the testing set
trainX, testX, trainY, testY = train_test_split(X, Y, test_size = 0.25)

# we need to turn the labels into 1-hot representations
trainY = to_categorical(trainY)
testY = to_categorical(testY)

# fit the model to the training data
model = FeedforwardNeuralNetwork([64, 32, 16, 10])
model.fit(trainX, trainY, 1000, 10)

# print the classification performance
print("Training set accuracy")

predictedY = model.predict(trainX)
predictedY = predictedY.argmax(axis=1)

trainY = trainY.argmax(axis=1)

print(classification_report(trainY, predictedY))

print("Test set accuracy")

predictedY = model.predict(testX)
predictedY = predictedY.argmax(axis=1)

testY = testY.argmax(axis=1)

print(classification_report(testY, predictedY))

Almost all test images were labeled correctly, 97%! But let's take a look at the ones that were labeled wrong. Most look a bit ambiguous or messy, so it's not that surprising they were not classified correctly.

In [ ]:
import matplotlib.pyplot as plt

for i in range(testX.shape[0]):
    if testY[i] != predictedY[i]:
        print(i)
        print('The correct label is', testY[i])
        print('The predicted label is', predictedY[i])
        plt.matshow(16*testX[i].reshape(8,8), cmap='gray_r')

Some of these images are really hard to interpret, even as a human, so it is unsurprising the model has trouble with them.

### Example: Full-Size MNIST

Next, let's try to use 1000 images from the full-sized MNIST dataset of 28-by-28 grayscale images.

In [ ]:
from tensorflow.keras.datasets import mnist

In [ ]:
### CLASSIFY MNIST PICTURES

# create a dataset of 1000 MNIST images, reshaped as single vectors, and labels
data = mnist.load_data()

# The datapoints are in mnistData[0][0]
X = data[0][0][:1000].reshape([1000,28*28])
X = X/255.0

# The labels are in mnistData[0][1]
Y = data[0][1][:1000]

# randomly choose 75% of the data to be the training set and 25% for the testing set
trainX, testX, trainY, testY = train_test_split(X, Y, test_size = 0.25)

trainY = to_categorical(trainY)
testY = to_categorical(testY)

# fit the model to the training data
model = FeedforwardNeuralNetwork([784, 16, 10], 0.5)
model.fit(trainX, trainY, 1000, 100)

# print the classification performance
print("Training set accuracy")
predictedY = model.predict(trainX)
predictedY = predictedY.argmax(axis=1)

trainY = trainY.argmax(axis=1)
print(classification_report(trainY, predictedY))

print("Test set accuracy")
predictedY = model.predict(testX)
predictedY = predictedY.argmax(axis=1)

testY = testY.argmax(axis=1)
print(classification_report(testY, predictedY))

Here, we have an **overfitting** problem because the training accuracy is good, but test accuracy is significantly worse. This frequently happens when we use a small dataset. In this case, we used only 1000 images, but it still took a several minutes to run. If we used 10000 images, which would be a more ideal dataset, it would take 30-40 minutes!

This is no good... we need stochastic gradient descent to make this more useful on larger datasets.

### Speeding Up Learning: SGD and Cross-Entropy

We already know what gradient descent is: we run our model on training data, compute the loss function, find the gradient of the loss function, and update our parameters in the opposite direction of the gradient by a small amount; and repeat this until the model converges. The same general approach is used in nearly all neural networks.

In linear regression and logistic classification, we were computing the outputs and loss function for the entire dataset (usually multiple times) per iteration of gradient descent, making a weight update based on the gradient, and repeating.

The typical approach used in modern neural networks is **stochastic** gradient descent (SGD). SGD updates weights after processing a random sample, called a **mini-batch**, of datapoints: enough points to reduce the variance of using just one for more stable convergence of parameters but not so much that the computation is too expensive. In this way, we process a mini-batch for outputs, compute an approximate loss function and approximate gradients, update weights, and repeat.

### How large should our mini-batches be?

Typically, using $2^n$ for something like $n=5, 6, 7, 8$ because these values tend to be ideal for the linear algebra optimization libraries we use (NumPy, TensorFlow, etc). The mini-batch size is a hyperparameter, but it generally isn't one you need to tweak too much.

One exception is if you are using GPUs for computing: then, it's typically best to choose the largest power of 2 that allows a whole mini-batch to fit into GPU memory, which will result in the underlying linear algebra libraries working optimally.

### Implementing SGD

Once we implement SGD with backpropagation, we will have constructed a "vanilla" neural network, which is probably the simplest neural network that is of practical use.

First, let's import some libraries we will be using.

In [ ]:
# load some libraries
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from keras.datasets import mnist
from keras.utils import to_categorical

Next, we will write a class similar to the `FeedforwardNeuralNetwork` class we wrote above but upgraded to use SGD.

In [ ]:
class FeedforwardNeuralNetworkSGD:
    
    # input a vector [a, b, c, ...] with the number of nodes in each layer
    def __init__(self, layers, alpha = 0.1, batchSize = 32):
        # list of weight matrices between layers
        self.W = []
        
        # network architecture will be a vector of numbers of nodes for each layer
        self.layers = layers
        
        # learning rate
        self.alpha = alpha
        
        # batch size
        self.batchSize = batchSize
        
        # initialize the weights (randomly) -- this is our initial guess for gradient descent
        
        # initialize the weights between layers (up to the next-to-last one) as normal random variables
        for i in np.arange(0, len(layers) - 2):
            self.W.append(np.random.randn(layers[i] + 1, layers[i + 1] + 1))
            
        # initialize weights between the last two layers (we don't want bias for the last one)
        self.W.append(np.random.randn(layers[-2] + 1, layers[-1]))
        
    # define the sigmoid activation
    def sigmoid(self, x):
        return 1.0 / (1 + np.exp(-x))
    
    # define the sigmoid derivative (where z is the output of a sigmoid)
    def sigmoidDerivative(self, z):
        return z * (1 - z)
    
    # get a new mini-batch of data from the dataset
    def getNextBatch(self, X, y, batchSize):
        for i in np.arange(0, X.shape[0], batchSize):
            # yield returns a generator, which can continue where it left off on later calls
            # of the function
            yield (X[i:i + batchSize], y[i:i + batchSize])
    
    # fit the model
    def fit(self, X, y, epochs = 10000, update = 1000):
        # add a column of ones to the end of X
        X = np.hstack((X, np.ones([X.shape[0],1])))

        for epoch in range(epochs):
            
            # randomize the examples
            p = np.arange(0, X.shape[0])
            np.random.shuffle(p)
            X = X[p]
            y = y[p]

            # feed forward, backprop, and weight update
            for (x, target) in self.getNextBatch(X, y, self.batchSize):
                
                # make a list of output activations from the first layer
                # (just the original x values)
                A = [np.atleast_2d(x)]
                
                # feed forward
                for layer in range(len(self.W)):
                    
                    # feed through one layer and apply sigmoid activation
                    net = A[layer].dot(self.W[layer])
                    out = self.sigmoid(net)
                    
                    # add our network output to the list of activations
                    A.append(out)
                    
                # backpropagation (coming soon!)
                error = A[-1] - target
                
                D = [error * self.sigmoidDerivative(A[-1])]
                
                # loop backwards over the layers to build up deltas
                for layer in np.arange(len(A) - 2, 0, -1):
                    delta = D[-1].dot(self.W[layer].T)
                    delta = delta * self.sigmoidDerivative(A[layer])
                    D.append(delta)
                    
                # reverse the deltas since we looped in reverse
                D = D[::-1]
                
                # weight update in each layer
                for layer in range(len(self.W)):
                    self.W[layer] -= self.alpha * A[layer].T.dot(D[layer])
                    
            # print an 
            if (epoch + 1) % update == 0:
                loss = self.computeLoss(X,y)
                print('Epoch =', epoch + 1, '\t loss =', loss)
                
    def predict(self, X, addOnes = True):
        # initialize data, be sure it's the right dimension
        p = np.atleast_2d(X)
        
        # add a column of 1s for bias
        if addOnes:
            p = np.hstack((p, np.ones([X.shape[0],1])))
        
        # feed forward!
        for layer in np.arange(0, len(self.W)):
            p = self.sigmoid(np.dot(p, self.W[layer]))
            
        return p
    
    def computeLoss(self, X, y):
        # initialize data, be sure it's the right dimension
        y = np.atleast_2d(y)
        
        # feed the datapoints through the network to get predicted outputs
        predictions = self.predict(X, addOnes = False)
        loss = np.sum((predictions - y)**2) / 2.0
        
        return loss

### Example: MNIST

As promised, this SGD neural net should run faster, so let's try to use the full 60,000 training images available in MNIST and 10,000 test images. (This is still a LOT of computation, using 70000 total 28-by-28 images.

In [ ]:
### CLASSIFY MNIST PICTURES

# load the full MNIST dataset: both data and labels
((trainX, trainY), (testX, testY)) = mnist.load_data()

# scale the data to values in [0,1]
trainX = trainX.astype('float32')/255.0
testX = testX.astype('float32')/255.0

# reshape the data
trainX = trainX.reshape([60000, 28*28])
testX = testX.reshape([10000, 28*28])

# convert the digits to one-hot vectors
trainY = to_categorical(trainY, 10)
testY = to_categorical(testY, 10)

# fit the model to the training data
model = FeedforwardNeuralNetworkSGD([784, 32, 16, 10], 0.5, 32)
model.fit(trainX, trainY, 100, 1)

# print the classification performance
print("Training set accuracy")
predictedY = model.predict(trainX)
predictedY = predictedY.argmax(axis=1)

trainY = trainY.argmax(axis=1)
print(classification_report(trainY, predictedY))

print("Test set accuracy")
predictedY = model.predict(testX)
predictedY = predictedY.argmax(axis=1)

testY = testY.argmax(axis=1)
print(classification_report(testY, predictedY))

Our test accuracy on MNIST jumped from mid-80\% previously to 95\% with our implementation using SGD and the full dataset!

### Cross-Entropy Loss Function

We will discuss in class, but the cross-entropy loss function can lead to faster training than the SSE we have used before, so we add it to the implementation.

In [ ]:
class FeedforwardNeuralNetworkSGD:
    
    # input a vector [a, b, c, ...] with the number of nodes in each layer
    def __init__(self, layers, alpha = 0.1, batchSize = 32, loss = 'sum-of-squares'):
        # list of weight matrices between layers
        self.W = []
        
        # network architecture will be a vector of numbers of nodes for each layer
        self.layers = layers
        
        # learning rate
        self.alpha = alpha
        
        # batch size
        self.batchSize = batchSize
        
        # loss function
        self.loss = loss
        
        # initialize the weights (randomly) -- this is our initial guess for gradient descent
        
        # initialize the weights between layers (up to the next-to-last one) as normal random variables
        for i in np.arange(0, len(layers) - 2):
            self.W.append(np.random.randn(layers[i] + 1, layers[i + 1] + 1))
            
        # initialize weights between the last two layers (we don't want bias for the last one)
        self.W.append(np.random.randn(layers[-2] + 1, layers[-1]))
        
    # define the sigmoid activation
    def sigmoid(self, x):
        return 1.0 / (1 + np.exp(-x))
    
    # define the sigmoid derivative (where z is the output of a sigmoid)
    def sigmoidDerivative(self, z):
        return z * (1 - z)
    
    # get a new mini-batch of data from the dataset
    def getNextBatch(self, X, y, batchSize):
        for i in np.arange(0, X.shape[0], batchSize):
            # yield returns a generator, which can continue where it left off on later calls
            # of the function
            yield (X[i:i + batchSize], y[i:i + batchSize])
    
    # fit the model
    def fit(self, X, y, epochs = 10000, update = 1000):
        # add a column of ones to the end of X
        X = np.hstack((X, np.ones([X.shape[0],1])))

        for epoch in range(epochs):
            
            # randomize the examples
            p = np.arange(0, X.shape[0])
            np.random.shuffle(p)
            X = X[p]
            y = y[p]

            # feed forward, backprop, and weight update
            for (x, target) in self.getNextBatch(X, y, self.batchSize):
                
                # make a list of output activations from the first layer
                # (just the original x values)
                A = [np.atleast_2d(x)]
                
                # feed forward
                for layer in range(len(self.W)):
                    
                    # feed through one layer and apply sigmoid activation
                    net = A[layer].dot(self.W[layer])
                    out = self.sigmoid(net)
                    
                    # add our network output to the list of activations
                    A.append(out)
                    
                # backpropagation (coming soon!)
                error = A[-1] - target
                
                if self.loss == 'sum-of-squares':
                    D = [error * self.sigmoidDerivative(A[-1])]
                    
                if self.loss == 'cross-entropy':
                    D = [error]
                    
                # loop backwards over the layers to build up deltas
                for layer in np.arange(len(A) - 2, 0, -1):
                    delta = D[-1].dot(self.W[layer].T)
                    delta = delta * self.sigmoidDerivative(A[layer])
                    D.append(delta)
                    
                # reverse the deltas since we looped in reverse
                D = D[::-1]
                
                # weight update in each layer
                for layer in range(len(self.W)):
                    self.W[layer] -= self.alpha * A[layer].T.dot(D[layer])
                    
            # print an 
            if (epoch + 1) % update == 0:
                loss = self.computeLoss(X,y)
                print('Epoch =', epoch + 1, '\t loss =', loss)
                
    def predict(self, X, addOnes = True):
        # initialize data, be sure it's the right dimension
        p = np.atleast_2d(X)
        
        # add a column of 1s for bias
        if addOnes:
            p = np.hstack((p, np.ones([X.shape[0],1])))
        
        # feed forward!
        for layer in np.arange(0, len(self.W)):
            p = self.sigmoid(np.dot(p, self.W[layer]))
            
        return p
    
    def computeLoss(self, X, y):
        # initialize data, be sure it's the right dimension
        y = np.atleast_2d(y)
        
        # feed the datapoints through the network to get predicted outputs
        predictions = self.predict(X, addOnes = False)
        
        # if the loss function is sum of squares, compute it
        if self.loss == 'sum-of-squares':
            loss = np.sum((predictions - y)**2) / 2.0
            
        # if the loss function is cross-entropy, compute it
        if self.loss == 'cross-entropy':
            loss = np.sum(np.nan_to_num(-y * np.log(predictions) - (1 - y) * np.log(1 - predictions)))
        
        return loss

In [ ]:
### CLASSIFY MNIST PICTURES

# load the full MNIST dataset: both data and labels
((trainX, trainY), (testX, testY)) = mnist.load_data()

# scale the data to values in [0,1]
trainX = trainX.astype('float32')/255.0
testX = testX.astype('float32')/255.0

# reshape the data
trainX = trainX.reshape([60000, 28*28])
testX = testX.reshape([10000, 28*28])

# convert the digits to one-hot vectors
trainY = to_categorical(trainY, 10)
testY = to_categorical(testY, 10)

# fit the model to the training data
model = FeedforwardNeuralNetworkSGD([784, 32, 16, 10], 0.1, 32, 'cross-entropy')
model.fit(trainX, trainY, 100, 1)

# print the classification performance
print("Training set accuracy")
predictedY = model.predict(trainX)
predictedY = predictedY.argmax(axis=1)

trainY = trainY.argmax(axis=1)
print(classification_report(trainY, predictedY))

print("Test set accuracy")
predictedY = model.predict(testX)
predictedY = predictedY.argmax(axis=1)

testY = testY.argmax(axis=1)
print(classification_report(testY, predictedY))

We get similar to slightly better results here.